In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# NTA-IDS Main Pipeline\n", "Network Traffic Analysis for Intrusion Detection"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('..')\n",
    "\n",
    "from src.preprocess import preprocess\n",
    "from src.features import select_features, engineer_features\n",
    "from src.dimensionality import apply_pca, train_autoencoder\n",
    "from src.models.random_forest import train_random_forest, evaluate as rf_evaluate, save_model as rf_save\n",
    "from src.models.svm import train_svm, evaluate as svm_evaluate, save_model as svm_save\n",
    "from src.models.lstm import train_lstm, evaluate as lstm_evaluate, save_model as lstm_save\n",
    "from src.ensemble import run_ensemble\n",
    "from src.evaluate import compute_metrics, plot_confusion_matrix, plot_training_history, compare_models\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "print('All imports OK')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 1 — Load and preprocess data"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Point this to your dataset folder\n",
    "# For CICIDS2017: '../data/raw/cicids2017'\n",
    "# For UNSW-NB15:  '../data/raw/unsw'\n",
    "\n",
    "DATA_PATH = '../data/raw/cicids2017'\n",
    "LABEL_COL = 'label'\n",
    "\n",
    "X_train, X_test, y_train, y_test, scaler, le = preprocess(\n",
    "    folder_path=DATA_PATH,\n",
    "    label_col=LABEL_COL,\n",
    "    test_size=0.2,\n",
    "    use_smote=True\n",
    ")\n",
    "\n",
    "label_names = list(le.classes_)\n",
    "num_classes = len(label_names)\n",
    "print(f'Classes: {label_names}')\n",
    "print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 2 — Dimensionality reduction (PCA)"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train_pca, X_test_pca, pca = apply_pca(X_train, X_test, variance_threshold=0.95)\n",
    "print(f'After PCA: {X_train_pca.shape}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 3 — Train Random Forest"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "rf_model = train_random_forest(X_train_pca, y_train)\n",
    "rf_pred, rf_cm = rf_evaluate(rf_model, X_test_pca, y_test, label_names)\n",
    "rf_metrics = compute_metrics(y_test, rf_pred, model_name='Random Forest')\n",
    "plot_confusion_matrix(y_test, rf_pred, label_names, model_name='Random Forest')\n",
    "rf_save(rf_model)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 4 — Train SVM"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "svm_model = train_svm(X_train_pca, y_train)\n",
    "svm_pred, svm_cm = svm_evaluate(svm_model, X_test_pca, y_test, label_names)\n",
    "svm_metrics = compute_metrics(y_test, svm_pred, model_name='SVM')\n",
    "plot_confusion_matrix(y_test, svm_pred, label_names, model_name='SVM')\n",
    "svm_save(svm_model)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 5 — Train LSTM"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "lstm_model, history = train_lstm(X_train_pca, y_train, X_test_pca, y_test, num_classes)\n",
    "lstm_pred, lstm_cm = lstm_evaluate(lstm_model, X_test_pca, y_test, label_names)\n",
    "lstm_metrics = compute_metrics(y_test, lstm_pred, model_name='LSTM')\n",
    "plot_confusion_matrix(y_test, lstm_pred, label_names, model_name='LSTM')\n",
    "plot_training_history(history)\n",
    "lstm_save(lstm_model)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 6 — Ensemble"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "hard_pred, weighted_pred = run_ensemble(\n",
    "    rf_model, svm_model, lstm_model,\n",
    "    X_test_pca, y_test,\n",
    "    weights=(0.3, 0.2, 0.5),\n",
    "    label_names=label_names\n",
    ")\n",
    "hard_metrics     = compute_metrics(y_test, hard_pred,     model_name='Hard Vote')\n",
    "weighted_metrics = compute_metrics(y_test, weighted_pred, model_name='Weighted Vote')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 7 — Compare all models"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "all_metrics = [rf_metrics, svm_metrics, lstm_metrics, hard_metrics, weighted_metrics]\n",
    "compare_models(all_metrics)\n",
    "\n",
    "import pandas as pd\n",
    "pd.DataFrame(all_metrics).set_index('model')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}